# 🛡️ ICS Network Anomaly Detection
## Notebook 02: Model Training & Evaluation

**Objective**: Train machine learning models to detect cyberattacks on Industrial Control Systems

**Models**:
- XGBoost Classifier (Primary)
- Random Forest (Comparison)
- Ensemble (Combined)

**Target**: Binary classification (Normal vs Attack)

**Applications**: Schneider Electric & Yokogawa OT/ICS Security

In [ ]:
# Import libraries
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
import joblib

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("✅ Libraries imported successfully!")

## 1️⃣ Load Engineered Features

In [ ]:
# Check if features exist
features_path = Path('../data/processed/ics_features.csv')
labels_path = Path('../data/processed/ics_labels.csv')

if not features_path.exists() or not labels_path.exists():
    print("⚠️  Features not found. Running feature engineering...")
    
    # Run feature engineering
    from ics_feature_engineer import ICSFeatureEngineer
    
    # Load raw data
    raw_data = pd.read_csv('../data/raw/kaggle/icssim/Dataset.csv', low_memory=False)
    
    # Create features
    engineer = ICSFeatureEngineer(random_seed=RANDOM_SEED)
    features, labels = engineer.create_all_features(raw_data)
    
    # Save
    engineer.save_features(features, labels, Path('../data/processed'))
else:
    print("📂 Loading engineered features...")
    features = pd.read_csv(features_path)
    labels = pd.read_csv(labels_path)['label']

print(f"\n✅ Loaded {len(features):,} samples")
print(f"   Features: {len(features.columns)}")
print(f"   Normal: {(labels == 0).sum():,}")
print(f"   Attack: {(labels == 1).sum():,}")

## 2️⃣ Prepare Training Data

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, 
    test_size=0.2, 
    random_state=RANDOM_SEED,
    stratify=labels
)

print("📊 Data Split:")
print(f"   Training: {len(X_train):,} samples")
print(f"   Testing: {len(X_test):,} samples")
print(f"\n   Train - Normal: {(y_train == 0).sum():,}, Attack: {(y_train == 1).sum():,}")
print(f"   Test  - Normal: {(y_test == 0).sum():,}, Attack: {(y_test == 1).sum():,}")

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled using StandardScaler")

# Convert back to DataFrame for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

## 3️⃣ Train XGBoost Model

In [ ]:
print("🚀 Training XGBoost Classifier...\n")

# XGBoost parameters
xgb_params = {
    'n_estimators': 200,
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': RANDOM_SEED,
    'n_jobs': -1,
    'eval_metric': 'logloss'
}

# Train model
xgb_model = xgb.XGBClassifier(**xgb_params)
xgb_model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=50
)

print("\n✅ XGBoost training completed!")

## 4️⃣ Train Random Forest (for comparison)

In [ ]:
print("🌲 Training Random Forest Classifier...\n")

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)

print("✅ Random Forest training completed!")

## 5️⃣ Make Predictions

In [ ]:
# Predictions
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_pred_rf = rf_model.predict(X_test_scaled)

# Probability predictions (for ROC curve)
y_pred_proba_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

print("✅ Predictions generated")

## 6️⃣ Evaluate Models

In [ ]:
def evaluate_model(y_true, y_pred, y_pred_proba, model_name):
    """Comprehensive model evaluation."""
    print("\n" + "="*80)
    print(f"{model_name} PERFORMANCE")
    print("="*80)
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred_proba)
    
    print(f"\n📊 Classification Metrics:")
    print(f"   Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"   Precision: {precision:.4f} ({precision*100:.2f}%)")
    print(f"   Recall:    {recall:.4f} ({recall*100:.2f}%)")
    print(f"   F1-Score:  {f1:.4f}")
    print(f"   ROC-AUC:   {auc:.4f}")
    
    # False positive/negative rates
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    
    print(f"\n🎯 Detection Rates:")
    print(f"   True Positives:  {tp:,}")
    print(f"   True Negatives:  {tn:,}")
    print(f"   False Positives: {fp:,} (FPR: {fpr*100:.2f}%)")
    print(f"   False Negatives: {fn:,} (FNR: {fnr*100:.2f}%)")
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'fpr': fpr,
        'fnr': fnr
    }

# Evaluate both models
xgb_metrics = evaluate_model(y_test, y_pred_xgb, y_pred_proba_xgb, "XGBoost")
rf_metrics = evaluate_model(y_test, y_pred_rf, y_pred_proba_rf, "Random Forest")

## 7️⃣ Visualize Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# XGBoost confusion matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
axes[0].set_title('XGBoost Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Random Forest confusion matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
axes[1].set_title('Random Forest Confusion Matrix', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('../results/confusion_matrices.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Saved: results/confusion_matrices.png")

## 8️⃣ ROC Curves

In [ ]:
# Calculate ROC curves
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_pred_proba_xgb)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)

# Plot
plt.figure(figsize=(10, 7))
plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {xgb_metrics["auc"]:.4f})', 
         linewidth=2, color='blue')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {rf_metrics["auc"]:.4f})', 
         linewidth=2, color='green')
plt.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random Classifier')

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - ICS Attack Detection', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/roc_curves.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Saved: results/roc_curves.png")

## 9️⃣ Feature Importance Analysis

In [ ]:
# Get feature importance from XGBoost
feature_importance = pd.DataFrame({
    'feature': features.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("🔝 Top 15 Most Important Features:")
print(feature_importance.head(15).to_string(index=False))

# Visualize
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'].values, 
         color='steelblue', edgecolor='black', alpha=0.7)
plt.yticks(range(len(top_features)), top_features['feature'].values)
plt.xlabel('Importance Score', fontsize=12)
plt.title('Top 15 Features for ICS Attack Detection', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Saved: results/feature_importance.png")

## 🔟 Model Comparison

In [ ]:
# Compare models
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'FPR', 'FNR'],
    'XGBoost': [
        xgb_metrics['accuracy'],
        xgb_metrics['precision'],
        xgb_metrics['recall'],
        xgb_metrics['f1'],
        xgb_metrics['auc'],
        xgb_metrics['fpr'],
        xgb_metrics['fnr']
    ],
    'Random Forest': [
        rf_metrics['accuracy'],
        rf_metrics['precision'],
        rf_metrics['recall'],
        rf_metrics['f1'],
        rf_metrics['auc'],
        rf_metrics['fpr'],
        rf_metrics['fnr']
    ]
})

comparison['Difference'] = comparison['XGBoost'] - comparison['Random Forest']
comparison['Winner'] = comparison.apply(
    lambda row: 'XGBoost' if row['Difference'] > 0 else 
                ('Random Forest' if row['Difference'] < 0 else 'Tie'), 
    axis=1
)

print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)
print(comparison.to_string(index=False))
print("="*80)

In [ ]:
# Visualize comparison
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
comparison_subset = comparison[comparison['Metric'].isin(metrics_to_plot)]

x = np.arange(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, comparison_subset['XGBoost'], width, 
               label='XGBoost', color='steelblue', edgecolor='black', alpha=0.8)
bars2 = ax.bar(x + width/2, comparison_subset['Random Forest'], width,
               label='Random Forest', color='forestgreen', edgecolor='black', alpha=0.8)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim([0.5, 1.0])
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../results/model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Saved: results/model_comparison.png")

## 1️⃣1️⃣ Save Models

In [ ]:
# Create models directory
models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)

# Save models
joblib.dump(xgb_model, models_dir / 'xgboost_ics_detector.pkl')
joblib.dump(rf_model, models_dir / 'random_forest_ics_detector.pkl')
joblib.dump(scaler, models_dir / 'feature_scaler.pkl')

print("💾 Models Saved:")
print("   ✅ models/xgboost_ics_detector.pkl")
print("   ✅ models/random_forest_ics_detector.pkl")
print("   ✅ models/feature_scaler.pkl")

# Save feature names
with open(models_dir / 'feature_names.txt', 'w') as f:
    f.write('\n'.join(features.columns))
print("   ✅ models/feature_names.txt")

## 1️⃣2️⃣ Save Predictions & Metrics

In [ ]:
# Save predictions
predictions_df = pd.DataFrame({
    'actual': y_test,
    'predicted_xgb': y_pred_xgb,
    'probability_xgb': y_pred_proba_xgb,
    'predicted_rf': y_pred_rf,
    'probability_rf': y_pred_proba_rf
})

predictions_df.to_csv('../results/test_predictions.csv', index=False)
print("✅ Saved: results/test_predictions.csv")

# Save metrics
comparison.to_csv('../results/model_metrics.csv', index=False)
print("✅ Saved: results/model_metrics.csv")

# Save feature importance
feature_importance.to_csv('../results/feature_importance.csv', index=False)
print("✅ Saved: results/feature_importance.csv")

## 📊 Final Summary

In [ ]:
print("\n" + "="*80)
print("ICS ANOMALY DETECTION - TRAINING COMPLETE")
print("="*80)

print("\n🎯 Best Model: XGBoost" if xgb_metrics['f1'] > rf_metrics['f1'] else "\n🎯 Best Model: Random Forest")

print("\n📊 Final Performance (XGBoost):")
print(f"   Accuracy:  {xgb_metrics['accuracy']*100:.2f}%")
print(f"   Precision: {xgb_metrics['precision']*100:.2f}%")
print(f"   Recall:    {xgb_metrics['recall']*100:.2f}%")
print(f"   F1-Score:  {xgb_metrics['f1']:.4f}")
print(f"   ROC-AUC:   {xgb_metrics['auc']:.4f}")

print("\n🏭 Industrial Applications:")
print("   ✅ Schneider Electric: PLC/SCADA attack detection")
print("   ✅ Yokogawa: DCS security monitoring")
print("   ✅ Real-time network traffic analysis")
print("   ✅ IEC 62443 compliance automation")

print("\n💾 Saved Artifacts:")
print("   • Trained models (XGBoost, Random Forest)")
print("   • Feature scaler")
print("   • Test predictions")
print("   • Performance metrics")
print("   • Visualizations (5 plots)")

print("\n✅ Model training completed successfully!")
print("➡️  Next: Deploy model or create SHAP explanations")
print("="*80)